In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler

In [56]:
train = pd.read_csv("../data/processed/train.csv")
val = pd.read_csv("../data/processed/val.csv")
test = pd.read_csv("../data/processed/test.csv")

In [57]:
train["Hour"] = (train["Time"] % 86_400) / 3600
val["Hour"] = (val["Time"] % 86_400) / 3600
test["Hour"] = (test["Time"] % 86_400) / 3600


train["Hour_sin"] = np.sin(2 * np.pi * train["Hour"] / 24)
train["Hour_cos"] = np.cos(2 * np.pi * train["Hour"] / 24)

val["Hour_sin"] = np.sin(2 * np.pi * val["Hour"] / 24)
val["Hour_cos"] = np.cos(2 * np.pi * val["Hour"] / 24)

test["Hour_sin"] = np.sin(2 * np.pi * test["Hour"] / 24)
test["Hour_cos"] = np.cos(2 * np.pi * test["Hour"] / 24)

In [58]:
train = train.drop(columns=["Time", "Hour"])
val = val.drop(columns=["Time", "Hour"])
test = test.drop(columns=["Time", "Hour"])



In [59]:
train["Amount_log"] = np.log1p(train["Amount"])
val["Amount_log"] = np.log1p(val["Amount"])
test["Amount_log"] = np.log1p(test["Amount"])

In [60]:
train = train.drop(columns=["Amount"])
val = val.drop(columns=["Amount"])
test = test.drop(columns=["Amount"])

In [61]:
cols_to_scale = [col for col in train.columns.to_list() if col not in
                 ["Hour_sin","Hour_cos","Class"]]

X_train = train.drop(columns=["Class"])
y_train = train["Class"]

X_val = val.drop(columns=["Class"])
y_val = val["Class"]

X_test = test.drop(columns=["Class"])
y_test = test["Class"]

transformer = ColumnTransformer(
    transformers=[
        ("robust_scaling", RobustScaler(), cols_to_scale)
    ],
    remainder="passthrough"
)

transformer.set_output(transform="pandas")

X_train = transformer.fit_transform(X_train)
X_val = transformer.transform(X_val)
X_test = transformer.transform(X_test)

X_train.columns = [col.split("__")[-1] for col in X_train.columns]
X_val.columns = [col.split("__")[-1] for col in X_val.columns]
X_test.columns = [col.split("__")[-1] for col in X_test.columns]

In [62]:
print(X_train.describe())

                  V1             V2             V3             V4  \
count  198277.000000  198277.000000  198277.000000  198277.000000   
mean       -0.004001      -0.050485      -0.090215       0.009959   
std         0.860440       1.162391       0.762747       0.880866   
min       -18.157575     -45.270718     -25.342989      -3.518568   
25%        -0.419444      -0.475121      -0.558843      -0.519533   
50%         0.000000       0.000000       0.000000       0.000000   
75%         0.580556       0.524879       0.441157       0.480467   
max         1.091211      13.451819       4.805067      10.573458   

                  V5             V6             V7             V8  \
count  198277.000000  198277.000000  198277.000000  198277.000000   
mean        0.044797       0.235298      -0.027070      -0.044514   
std         1.042942       1.143267       1.054903       2.170001   
min       -87.131864     -19.939280     -23.643933     -95.213097   
25%        -0.487116      -0.4251

In [68]:
datasets = {
    "X_train": X_train,
    "X_val": X_val,
    "X_test": X_test,
    "y_train": y_train,
    "y_val": y_val,
    "y_test": y_test,
}

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)
for name, df in datasets.items():
    df.to_csv(output_dir/f"{name}.csv", index=False)